In [8]:
import pandas as pd
import numpy as np

# Afficher toutes les colonnes qu'on va créer
pd.set_option('display.max_columns', 50)

In [9]:
# Chargement des données brutes
train = pd.read_csv('../data/raw/train.csv', parse_dates=['date'])
test = pd.read_csv('../data/raw/test.csv', parse_dates=['date'])

# Concaténation : test aura sales = NaN
df = pd.concat([train, test], sort=False).reset_index(drop=True)

# Tri chronologique indispensable pour les séries temporelles !
df = df.sort_values(by=['store', 'item', 'date']).reset_index(drop=True)

print(f"Dimensions totales (Train + Test) : {df.shape[0]:,} lignes, {df.shape[1]} colonnes".replace(',', ' '))
print(f"Période complète : du {df['date'].min().strftime('%Y-%m-%d')} au {df['date'].max().strftime('%Y-%m-%d')}")
display(df.head())

Dimensions totales (Train + Test) : 958 000 lignes  5 colonnes
Période complète : du 2013-01-01 au 2018-03-31


,date,store,item,sales,id
0,2013-01-01,1,1,13.0,NaN
1,2013-01-02,1,1,11.0,NaN
2,2013-01-03,1,1,14.0,NaN
3,2013-01-04,1,1,13.0,NaN
4,2013-01-05,1,1,10.0,NaN


In [10]:
def create_date_features(df):
    """
    Extrait les composantes de calendrier à partir de la colonne date
    """
    df_feat = df.copy()
    
    # 1. Composantes directes
    df_feat['year'] = df_feat['date'].dt.year
    df_feat['month'] = df_feat['date'].dt.month
    df_feat['day'] = df_feat['date'].dt.day
    df_feat['dayofweek'] = df_feat['date'].dt.dayofweek  # 0 = Lundi, 6 = Dimanche
    df_feat['dayofyear'] = df_feat['date'].dt.dayofyear
    df_feat['weekofyear'] = df_feat['date'].dt.isocalendar().week.astype(int)
    
    # 2. Indicateurs booléens (0 ou 1) utiles pour le commerce
    df_feat['is_weekend'] = df_feat['dayofweek'].isin([5, 6]).astype(int)      # Samedi (5) et Dimanche (6)
    df_feat['is_month_start'] = df_feat['date'].dt.is_month_start.astype(int) # 1er jour du mois
    df_feat['is_month_end'] = df_feat['date'].dt.is_month_end.astype(int)     # Fin de mois (paie, etc.)
    
    return df_feat

# Application de la fonction
df = create_date_features(df)

# Vérification sur les 5 premières lignes
display(df[['date', 'store', 'item', 'year', 'month', 'dayofweek', 'dayofyear', 'is_weekend']].head())

,date,store,item,year,month,dayofweek,dayofyear,is_weekend
0,2013-01-01,1,1,2013,1,1,1,0
1,2013-01-02,1,1,2013,1,2,2,0
2,2013-01-03,1,1,2013,1,3,3,0
3,2013-01-04,1,1,2013,1,4,4,0
4,2013-01-05,1,1,2013,1,5,5,1


In [11]:
def create_lags(df, lags=[91, 98, 105, 112, 119, 126, 182, 364]):
    """
    Crée des variables de retard (lags) par couple store x item.
    Le décalage minimal est de 91 jours (> horizon de 90 jours du test).
    """
    df_feat = df.copy()
    
    # On groupe par magasin et produit, puis on décale les ventes dans le temps
    for lag in lags:
        df_feat[f'sales_lag_{lag}'] = (
            df_feat.groupby(['store', 'item'])['sales']
            .shift(lag)
        )
        
    return df_feat

# Application des lags
df = create_lags(df)

# Aperçu des colonnes créées
lag_cols = [c for c in df.columns if 'lag' in c]
print(f"Colonnes de lags créées ({len(lag_cols)}) : {lag_cols}")
display(df[['date', 'store', 'item', 'sales'] + lag_cols[:4]].tail(10))

Colonnes de lags créées (8) : ['sales_lag_91', 'sales_lag_98', 'sales_lag_105', 'sales_lag_112', 'sales_lag_119', 'sales_lag_126', 'sales_lag_182', 'sales_lag_364']


,date,store,item,sales,sales_lag_91,sales_lag_98,sales_lag_105,sales_lag_112
957990,2018-03-22,10,50,NaN,63.0,72.0,66.0,73.0
957991,2018-03-23,10,50,NaN,75.0,72.0,67.0,66.0
957992,2018-03-24,10,50,NaN,70.0,52.0,69.0,49.0
957993,2018-03-25,10,50,NaN,76.0,86.0,69.0,75.0
957994,2018-03-26,10,50,NaN,51.0,53.0,54.0,54.0
957995,2018-03-27,10,50,NaN,41.0,54.0,67.0,68.0
957996,2018-03-28,10,50,NaN,63.0,51.0,67.0,60.0
957997,2018-03-29,10,50,NaN,59.0,63.0,72.0,66.0
957998,2018-03-30,10,50,NaN,74.0,75.0,72.0,67.0
957999,2018-03-31,10,50,NaN,62.0,70.0,52.0,69.0


In [12]:
def create_rolling_features(df):
    """
    Calcule des moyennes mobiles et l'écart-type sur les ventes passées (à partir de lag 91)
    """
    df_feat = df.copy()
    
    # Base décalée de 91 jours pour chaque série
    grouped = df_feat.groupby(['store', 'item'])['sales']
    
    # Moyennes mobiles sur différentes fenêtres
    windows = [7, 14, 30, 90]
    for w in windows:
        df_feat[f'rolling_mean_{w}'] = (
            grouped.shift(91)
            .rolling(window=w)
            .mean()
        )
        
    # Écart-type (volatilité des ventes sur 30 jours)
    df_feat['rolling_std_30'] = (
        grouped.shift(91)
        .rolling(window=30)
        .std()
    )
    
    return df_feat

# Application des rolling features
df = create_rolling_features(df)

rolling_cols = [c for c in df.columns if 'rolling' in c]
print(f"Colonnes rolling créées ({len(rolling_cols)}) : {rolling_cols}")
display(df[['date', 'store', 'item', 'sales'] + rolling_cols].tail(5))

Colonnes rolling créées (5) : ['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_mean_90', 'rolling_std_30']


,date,store,item,sales,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_mean_90,rolling_std_30
957995,2018-03-27,10,50,NaN,61.000000,63.071429,64.633333,77.811111,10.436948
957996,2018-03-28,10,50,NaN,62.714286,62.785714,64.233333,77.600000,10.254295
957997,2018-03-29,10,50,NaN,62.142857,61.857143,63.933333,77.255556,10.271935
957998,2018-03-30,10,50,NaN,62.000000,62.000000,63.866667,76.933333,10.197137
957999,2018-03-31,10,50,NaN,60.857143,62.714286,63.500000,76.522222,10.054164


In [13]:
# 1. Cible en log (np.log1p calcule log(1 + x))
df['sales_log'] = np.log1p(df['sales'])

# 2. Séparation Train et Test
# On garde le train à partir de 2014 (une fois que tous les lags 364j sont disponibles)
train_features = df[(df['date'] >= '2014-01-01') & (df['sales'].notnull())].copy()
test_features = df[df['sales'].isnull()].copy()

# On supprime la colonne temporaire id du train si présente
if 'id' in train_features.columns:
    train_features = train_features.drop(columns=['id'])

print("--- Données finales prêtes pour la modélisation ---")
print(f"Train prêt : {train_features.shape[0]:,} lignes, {train_features.shape[1]} colonnes (de {train_features['date'].min().strftime('%Y-%m-%d')} à {train_features['date'].max().strftime('%Y-%m-%d')})".replace(',', ' '))
print(f"Test prêt  : {test_features.shape[0]:,} lignes, {test_features.shape[1]} colonnes (de {test_features['date'].min().strftime('%Y-%m-%d')} à {test_features['date'].max().strftime('%Y-%m-%d')})".replace(',', ' '))

# Contrôle : reste-t-il des valeurs nulles dans nos features d'entraînement ?
features_list = [c for c in train_features.columns if c not in ['date', 'sales', 'sales_log', 'id']]
print(f"\nNombre total de features créées : {len(features_list)}")
print(f"Valeurs manquantes restantes dans les features Train : {train_features[features_list].isnull().sum().sum()}")

--- Données finales prêtes pour la modélisation ---
Train prêt : 730 500 lignes  27 colonnes (de 2014-01-01 à 2017-12-31)
Test prêt  : 45 000 lignes  28 colonnes (de 2018-01-01 à 2018-03-31)

Nombre total de features créées : 24
Valeurs manquantes restantes dans les features Train : 0


In [14]:
# Sauvegarde au format CSV dans data/processed
train_features.to_csv('../data/processed/train_features.csv', index=False)
test_features.to_csv('../data/processed/test_features.csv', index=False)

print("✅ Fichiers sauvegardés avec succès dans data/processed/ !")
print("- ../data/processed/train_features.csv")
print("- ../data/processed/test_features.csv")

✅ Fichiers sauvegardés avec succès dans data/processed/ !
- ../data/processed/train_features.csv
- ../data/processed/test_features.csv


## Synthèse du Feature Engineering & Sauvegarde

### Ce qui a été accompli :
1. **Concaténation Train + Test :** Permet de calculer l'historique complet pour la période de test sans rupture.
2. **Variables de calendrier (Date Features) :** `year`, `month`, `day`, `dayofweek`, `dayofyear`, `weekofyear`, `is_weekend`, `is_month_start`, `is_month_end`.
3. **Variables de retard (Lags) anti-leakage :** Décalages de 91, 98, 105, 112, 119, 126, 182 et 364 jours. Le seuil minimum de 91 jours respecte scrupuleusement l'horizon de 90 jours du test.
4. **Moyennes mobiles (Rolling Features) :** Moyennes sur 7, 14, 30, 90 jours et écart-type (volatilité) sur 30 jours, toutes basées sur le décalage de sécurité.
5. **Transformation de la cible :** Ajout de `sales_log = np.log1p(sales)` pour symétriser la distribution.
6. **Nettoyage :** Filtrage à partir de 2014 pour éliminer l'année d'initialisation (zéro NaN restant dans le jeu d'entraînement).
7. **Exportation :** Données sauvegardées dans `data/processed/train_features.csv` et `data/processed/test_features.csv`.